### Setup & Konfiguration

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

# Notebook-Rendering
pio.renderers.default = "notebook_connected"

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


### Parser importieren

In [ ]:
PROJECT_ROOT = Path().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"
DATA_PATH = PROJECT_ROOT / "data/raw"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

print("Projekt-Root:", PROJECT_ROOT)
print("SRC-Pfad:", SRC_PATH)

from core.marc21_parser_full import parse_dnb_theses

### Parameterzelle

In [ ]:
FILE_NAME = "dnb-all_hochschulschriften_dnbmarc.mrc.xml"
LIMIT = 5000  # None = gesamte Datei
VERBOSE = True

FILE_PATH = DATA_PATH / FILE_NAME

assert FILE_PATH.exists(), f"Datei nicht gefunden: {FILE_PATH}"

### Daten laden

In [ ]:
df_raw = parse_dnb_theses(FILE_PATH, limit=LIMIT, verbose=VERBOSE)

print(f"DataFrame geladen: {df_raw.shape[0]} Datensätze, {df_raw.shape[1]} Spalten")
display(df_raw.head())

### Explorations-DataFrame vorbereiten

In [ ]:
df = df_raw.copy()

# Sicherstellen, dass Listenfelder wirklich Listen sind
LIST_COLUMNS = [
    "082_list",
    "083_list",
    "084_list",
    "subjects",
    "idn_list",
]

for col in LIST_COLUMNS:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: x if isinstance(x, list) else [])


### Fehlende Notationsfelder

In [ ]:
NOTATION_FIELDS = ["082_list", "083_list", "084_list"]

missing_percent = (
    df[NOTATION_FIELDS]
    .apply(lambda col: col.apply(lambda x: len(x) == 0).mean() * 100)
    .round(2)
)

print("Fehlende Notationsfelder (%):")
display(missing_percent)

df["notation_missing"] = df[NOTATION_FIELDS].apply(
    lambda row: all(len(v) == 0 for v in row),
    axis=1,
)

print(
    f"Datensätze ohne Notation: "
    f"{df['notation_missing'].sum()} / {len(df)} "
    f"({df['notation_missing'].mean()*100:.2f}%)"
)


### Verteilung der Anzahl Notationen pro Datensatz

In [ ]:
# Anzahl Notationen pro Record
df["notation_count"] = df[NOTATION_FIELDS].apply(
    lambda row: sum(len(v) for v in row),
    axis=1,
)

fig = px.histogram(
    df,
    x="notation_count",
    nbins=20,
    title="Verteilung der Notationsanzahl pro Datensatz",
)

fig.show()

### DDC-Top-Level Analyse (optional, falls 082 genutzt wird)

In [ ]:
ddc_values = []

for entries in df["082_list"]:
    for entry in entries:
        if isinstance(entry, dict) and "a" in entry:
            ddc = entry["a"]
            if ddc:
                ddc_values.append(ddc[:3])  # erste 3 Stellen

counter = Counter(ddc_values)
top_ddc = pd.DataFrame(counter.most_common(20), columns=["DDC", "count"])

fig = px.bar(
    top_ddc,
    x="DDC",
    y="count",
    title="Top 20 DDC (erste 3 Stellen)",
)

fig.show()


### Sprachverteilung

In [ ]:
lang_dist = df["language"].value_counts().head(20)

fig = px.bar(
    lang_dist,
    title="Top Sprachen",
)

fig.show()


### Overview

In [ ]:

def preview_value(value, max_items=3):
    """
    Erzeugt eine kompakte Vorschau für skalare, Listen- oder Dict-Werte.
    """
    if isinstance(value, list):
        preview = []
        for v in value[:max_items]:
            if isinstance(v, dict):
                preview.append({k: v[k] for k in v})
            else:
                preview.append(v)
        return preview

    if isinstance(value, dict):
        return {k: value[k] for k in list(value.keys())[:max_items]}

    return value


def column_unique_count(series: pd.Series) -> int:
    """
    Robuste Unique-Zählung auch für Listen-/Dict-Spalten.
    """
    try:
        return series.dropna().nunique()
    except TypeError:
        return series.dropna().apply(lambda x: str(x)).nunique()


def column_preview(series: pd.Series, n=3):
    """
    Preview der ersten n nicht-null Werte.
    """
    values = series.dropna().head(n).tolist()
    return [preview_value(v) for v in values]


# -----------------------------------
# Overview DataFrame
# -----------------------------------

overview = pd.DataFrame({
    "dtype": df.dtypes,
    "non_null": df.notna().sum(),
})

overview["missing_n"] = df.isna().sum()
overview["missing_%"] = (df.isna().mean() * 100).round(2)

overview["unique_n"] = [
    column_unique_count(df[col])
    for col in df.columns
]

overview["preview"] = [
    column_preview(df[col])
    for col in df.columns
]

overview = overview.sort_values("missing_%", ascending=False)

display(overview)


### Anteil leerer Notationsfelder

In [ ]:

# inhaltlich fehlende Notationen statt NaN

NOTATION_FIELDS = ["082_list", "083_list", "084_list"]

empty_percent = {
    col: (df[col].apply(len) == 0).mean() * 100
    for col in NOTATION_FIELDS
}

empty_percent = pd.Series(empty_percent).round(2)

plt.figure(figsize=(6, 4))
sns.barplot(x=empty_percent.index, y=empty_percent.values)
plt.ylabel("Leere Felder (%)")
plt.title("Anteil Datensätze ohne Notation")
plt.ylim(0, 100)
plt.show()


### DDC-Hierarchie sinnvoll visualisieren

In [ ]:
df_ddc = df.copy()

# Explode nur wenn nicht leer
df_ddc = df_ddc[df_ddc["082_list"].apply(len) > 0]
df_ddc = df_ddc.explode("082_list")

# a-Feld extrahieren
df_ddc["082_a"] = df_ddc["082_list"].apply(
    lambda x: x.get("a") if isinstance(x, dict) else None
)

# Bereinigung
df_ddc = df_ddc[df_ddc["082_a"].notna()]
df_ddc["082_a"] = df_ddc["082_a"].str.strip()
df_ddc = df_ddc[df_ddc["082_a"] != ""]

# Hierarchie erzeugen
df_ddc["DDC_1"] = df_ddc["082_a"].str[0]
df_ddc["DDC_2"] = df_ddc["082_a"].str[:2]
df_ddc["DDC_3"] = df_ddc["082_a"].str[:3]


In [ ]:
print("Gesamte Datensätze:", len(df))
print("Nicht-leere 082_list:", (df["082_list"].apply(len) > 0).sum())

In [ ]:
DDC_MAIN = {
    "0": "Allgemeines",
    "1": "Philosophie",
    "2": "Religion",
    "3": "Sozialwissenschaften",
    "4": "Sprache",
    "5": "Naturwissenschaften",
    "6": "Technik",
    "7": "Kunst",
    "8": "Literatur",
    "9": "Geschichte"
}

df_ddc["DDC_1_label"] = df_ddc["DDC_1"].map(DDC_MAIN)

fig = px.sunburst(
    df_ddc,
    path=["DDC_1_label", "DDC_2", "DDC_3"],
    title="DDC-Hierarchie mit Hauptklassen"
)
fig.show()


### Häufigkeitsbasierte Filterung

In [ ]:
top_codes = df_ddc["DDC_3"].value_counts().head(20).index
df_ddc_top = df_ddc[df_ddc["DDC_3"].isin(top_codes)]


top_counts = df_ddc["DDC_3"].value_counts().head(20)

plt.figure(figsize=(8,6))
sns.barplot(
    x=top_counts.values,
    y=top_counts.index
)
plt.title("Top 20 DDC (3-stellig)")
plt.xlabel("Anzahl")
plt.ylabel("DDC")
plt.show()


### Übersicht Notationssysteme in 084$2 

In [ ]:
# Nur Records mit 084
df_084 = df[df["084_list"].apply(len) > 0].copy()

# Explode
df_084 = df_084.explode("084_list")

# $2 extrahieren
df_084["system"] = df_084["084_list"].apply(
    lambda d: d.get("2", "").strip().lower()
    if isinstance(d, dict) else ""
)

# Leere entfernen
df_084 = df_084[df_084["system"] != ""]

# Zählen
system_counts = df_084["system"].value_counts()

print("Gefundene Notationssysteme in 084$2:")
display(system_counts)



In [ ]:
# Nur Records mit System
df_084 = df[df["084_list"].apply(len) > 0].copy()
df_084 = df_084.explode("084_list")
df_084["system"] = df_084["084_list"].apply(
    lambda d: d.get("2", "").strip().lower() if isinstance(d, dict) else ""
)
df_084 = df_084[df_084["system"] != ""]

# Häufigkeiten in Prozent
system_percent = df_084["system"].value_counts(normalize=True).sort_values(ascending=False) * 100
system_percent = system_percent.round(1)

# Konsolenausgabe
print("Notationssysteme in 084$2 mit Prozentwerten:")
display(system_percent)

# Balkendiagramm
plt.figure(figsize=(10, 5))
sns.barplot(x=system_percent.index, y=system_percent.values, palette="magma")
plt.ylabel("Prozent (%)")
plt.xlabel("Notationssystem (084$2)")
plt.title("Verteilung der Notationssysteme in MARC 084$2")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### EDA für ML-Vorhersage von DDC

In [ ]:
#
# -------------------------------
# 0️⃣ Vorbereitung der Features
# -------------------------------

# --- 082_list_a aus 082_list extrahieren ---
df['082_list_a'] = df['082_list'].apply(
    lambda lst: [d.get('a','').strip() for d in lst if isinstance(d, dict) and d.get('a')]
)

# --- DDC fehlend markieren ---
df['ddc_missing'] = df[['082_list', '083_list', '084_list']].apply(
    lambda x: all(len(lst) == 0 for lst in x), axis=1
)

# --- Autorenanzahl berechnen ---
df['num_authors'] = df['author_name'].apply(
    lambda x: len(x) if isinstance(x, list) else (1 if isinstance(x, str) and x else 0)
)

# --- Titel-Längen ---
df['title_length'] = df['title'].fillna("").apply(len)
df['title_remainder_length'] = df['title_remainder'].fillna("").apply(len)

# --- Veröffentlichungsjahr numerisch ---
def parse_year(y):
    if isinstance(y, str):
        y = y.strip("[]")  # evtl. [1975] -> 1975
        try:
            return int(y)
        except:
            return None
    return None

df['publication_year_num'] = df['publication_year'].apply(parse_year)

# --- Sprache (erstes Element) ---
df['language_first'] = df['language'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'unknown'
)

# -------------------------------
# 1️⃣ Missing Data Übersicht
# -------------------------------

features = ['title', 'title_remainder', 'author_name', 'publication_year',
            'publisher', 'language', '082_list_a', 'ddc_missing']

missing_summary = df[features].isna().mean() * 100
print("Fehlende Werte (%) pro Feature:")
display(missing_summary)

plt.figure(figsize=(10,5))
sns.barplot(x=missing_summary.index, y=missing_summary.values)
plt.xticks(rotation=45, ha='right')
plt.ylabel("Fehlende Werte (%)")
plt.title("Missing Data Übersicht")
plt.tight_layout()
plt.show()

# -------------------------------
# 2️⃣ Textlängen analysieren
# -------------------------------

plt.figure(figsize=(12,4))
sns.histplot(df['title_length'], bins=50, kde=True)
plt.title("Längenverteilung der Titel")
plt.xlabel("Zeichen")
plt.show()

plt.figure(figsize=(12,4))
sns.histplot(df['title_remainder_length'], bins=50, kde=True)
plt.title("Längenverteilung der Titelremainder")
plt.xlabel("Zeichen")
plt.show()

# -------------------------------
# 3️⃣ Autorenverteilung
# -------------------------------

plt.figure(figsize=(10,4))
sns.histplot(df['num_authors'], bins=20, kde=False)
plt.title("Anzahl Autoren pro Datensatz")
plt.xlabel("Anzahl Autoren")
plt.show()

# -------------------------------
# 4️⃣ Veröffentlichungsjahre
# -------------------------------

plt.figure(figsize=(12,4))
sns.histplot(df['publication_year_num'].dropna(), bins=50, kde=False)
plt.title("Verteilung der Veröffentlichungsjahre")
plt.xlabel("Jahr")
plt.show()

# -------------------------------
# 5️⃣ Sprache der Doktorarbeiten
# -------------------------------

plt.figure(figsize=(10,4))
sns.countplot(y='language_first', data=df, order=df['language_first'].value_counts().index)
plt.title("Sprache der Doktorarbeiten")
plt.show()

# -------------------------------
# 6️⃣ DDC Verteilung (nur vorhandene)
# -----------------------


In [ ]:
# -----------------------------------
# 1️. Titel-Länge vs. DDC-Vorhandensein
# -----------------------------------
plt.figure(figsize=(10,5))
sns.boxplot(x='ddc_missing', y='title_length', data=df)
plt.xticks([0,1], ['DDC vorhanden', 'DDC fehlt'])  # optional, funktioniert nur wenn Boolean als 0/1
plt.xlabel("DDC-Vorhandensein")
plt.ylabel("Titel-Länge (Zeichen)")
plt.title("Titel-Länge in Abhängigkeit vom DDC-Vorhandensein")
plt.show()

# Bessere Alternative: Boolean direkt als Kategorie
plt.figure(figsize=(10,5))
sns.boxplot(x=df['ddc_missing'].map({False:'DDC vorhanden', True:'DDC fehlt'}),
            y='title_length',
            data=df)
plt.xlabel("DDC-Vorhandensein")
plt.ylabel("Titel-Länge (Zeichen)")
plt.title("Titel-Länge in Abhängigkeit vom DDC-Vorhandensein")
plt.show()

# -----------------------------------
# 2. Veröffentlichungsjahr vs. DDC-Vorhandensein
# -----------------------------------
plt.figure(figsize=(12,5))
sns.boxplot(x=df['ddc_missing'].map({False:'DDC vorhanden', True:'DDC fehlt'}),
            y='publication_year_num',
            data=df)
plt.xlabel("DDC-Vorhandensein")
plt.ylabel("Veröffentlichungsjahr")
plt.title("Veröffentlichungsjahr vs. DDC-Vorhandensein")
plt.show()


### DDC-Vergabe nach Jahrzehnt – gestapelte Balken

In [ ]:


# Zuerst in Zahlen umwandeln, Fehlerhafte Werte zu NaN
df['publication_year_num'] = pd.to_numeric(df['publication_year'], errors='coerce')

# Nur gültige Jahre behalten
df_plot = df[df['publication_year_num'].notnull()].copy()

# Jahrzehnt-Spalte erstellen
df_plot['decade'] = (df_plot['publication_year_num'] // 10).astype(int) * 10

# DDC-Präsenz ableiten
df_plot['ddc_present'] = ~df_plot['ddc_missing']

# Gruppieren nach Jahrzehnt
summary = df_plot.groupby('decade')['ddc_present'].agg(
    total='count',
    ddc_assigned='sum'
)
summary['ddc_missing'] = summary['total'] - summary['ddc_assigned']

# Gestapelte Balken
summary[['ddc_assigned', 'ddc_missing']].plot(
    kind='bar',
    stacked=True,
    figsize=(10,6),

)
plt.xlabel("Jahrzehnt")
plt.ylabel("Anzahl Thesen")
plt.title("DDC-Vergabe nach Jahrzehnt")
plt.xticks(rotation=0)
plt.legend(["DDC vorhanden","DDC fehlt"])
plt.show()


In [ ]:
# Gesamtanzahl aller Thesen
total_rows = len(df)

# Anzahl gültiger Jahre
valid_rows = len(df_plot)

# Anzahl der herausgenommenen Zeilen
removed_rows = total_rows - valid_rows

print(f"Insgesamt Thesen: {total_rows}")
print(f"Thesen mit gültigem Jahr: {valid_rows}")
print(f"Thesen ohne gültiges Jahr (entfernt): {removed_rows}")

